In [2]:
import os
import pytesseract
from PIL import Image, ImageTk
from nltk.tokenize import word_tokenize
import tkinter as tk
from tkinter import filedialog, scrolledtext, messagebox
import requests
import cv2
from docx import Document
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

# Set the path to the Tesseract executable (modify this based on your installation)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Set the TESSDATA_PREFIX environment variable
os.environ['TESSDATA_PREFIX'] = r'C:\Program Files\Tesseract-OCR\tessdata'


class WordMeaningApp:

    def get_word_meaning(self, word):
        try:
            response = requests.get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{word}")
            if response.status_code == 200:
                data = response.json()
                meaning = data[0]['meanings'][0]['definitions'][0]['definition']
                return meaning
            else:
                return "Meaning not found"
        except Exception as e:
            print(f"Error fetching meaning from online dictionary: {e}")
            return "Meaning not found"

    def extract_text_from_image(self, image_path):
        try:
            # Open the image using PIL (Python Imaging Library)
            image = Image.open(image_path)

            # Use pytesseract to extract text from the image
            extracted_text = pytesseract.image_to_string(image)

            return extracted_text
        except Exception as e:
            print(f"Error extracting text from image: {e}")
            return ""

    def __init__(self, master):
        self.master = master

        self.master.title("Word Meaning App")
        self.master.geometry("800x600")  # Set initial window size
        self.master.configure(bg="#f0f0f0")

        self.main_frame = tk.Frame(self.master)
        self.main_frame.pack(expand=True, fill='both')

        # Create a Label to display the image
        self.image_label = tk.Label(self.main_frame)
        self.image_label.pack(side=tk.LEFT, padx=10, pady=10)

        # Create a scrolled text widget to display the extracted words
        self.scrolled_text = scrolledtext.ScrolledText(self.master, wrap=tk.WORD, width=40, height=10)
        self.scrolled_text.pack(expand=True, fill='both')

        # Store the reference to the current meaning window
        self.current_meaning_window = None

        # Add an "Import Images" button
        self.import_button = tk.Button(self.master, text="Import Images", command=self.import_images)
        self.import_button.pack()

        # Add a "Access Camera" button
        self.camera_button = tk.Button(self.master, text="Access Camera", command=self.access_camera)
        self.camera_button.pack()

        # Add a "Capture Image" button
        self.capture_button = tk.Button(self.master, text="Capture Image", command=self.capture_image, state=tk.DISABLED)
        self.capture_button.pack()

        # Add "Previous" and "Next" buttons to navigate through images
        self.prev_button = tk.Button(self.master, text="Previous", command=self.show_previous_image)
        self.prev_button.pack(side=tk.LEFT, padx=5)

        self.next_button = tk.Button(self.master, text="Next", command=self.show_next_image)
        self.next_button.pack(side=tk.RIGHT, padx=5)

        # Add buttons for saving text as Word or PDF
        self.save_word_button = tk.Button(self.master, text="Save as Word", command=self.save_as_word, state=tk.DISABLED)
        self.save_word_button.pack()

        self.save_pdf_button = tk.Button(self.master, text="Save as PDF", command=self.save_as_pdf, state=tk.DISABLED)
        self.save_pdf_button.pack()

        # Initialize camera capture object
        self.cap = cv2.VideoCapture(0)
        self.camera_running = False

        # List to store paths of selected images
        self.selected_images = []
        self.current_image_index = -1

        # Variable to store extracted text
        self.extracted_text = ""

        # Call the update_camera method every 100 milliseconds
        self.master.after(100, self.update_camera)

    def display_image_on_label(self, image):
        # Resize the image to fit the label
        base_height = 300
        img_ratio = image.height / image.width
        new_width = int(base_height / img_ratio)
        image = image.resize((new_width, base_height), Image.Resampling.LANCZOS)

        # Convert the image to a format that Tkinter can use
        photo = ImageTk.PhotoImage(image)
        self.image_label.config(image=photo)
        self.image_label.image = photo  # Keep a reference to avoid garbage collection

    def import_images(self):
        # Prompt the user to select multiple image files
        file_paths = filedialog.askopenfilenames(filetypes=[("Image files", "*.png;*.jpg;*.jpeg;*.gif;*.bmp")])

        if file_paths:
            self.selected_images = list(file_paths)
            self.current_image_index = 0
            self.process_image(self.selected_images[self.current_image_index])

    def access_camera(self):
        self.camera_running = not self.camera_running
        if self.camera_running:
            self.capture_button.config(state=tk.NORMAL)
        else:
            self.capture_button.config(state=tk.DISABLED)

    def capture_image(self):
        # Capture an image from the camera
        ret, frame = self.cap.read()
        if ret:
            # Save the captured image
            captured_image_path = f"captured_image_{len(self.selected_images)}.png"
            cv2.imwrite(captured_image_path, frame)

            self.selected_images.append(captured_image_path)
            self.current_image_index = len(self.selected_images) - 1

            self.process_image(captured_image_path)

    def update_camera(self):
        # Continuously capture frames from the camera if it's running
        if self.camera_running:
            ret, frame = self.cap.read()

            # Convert the frame to RGB format
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Convert the frame to PIL Image
            image = Image.fromarray(frame_rgb)
            self.display_image_on_label(image)

        # Call the update_camera method again after 100 milliseconds
        self.master.after(100, self.update_camera)

    def process_image(self, image_path):
        # Clear existing text and meaning windows
        self.scrolled_text.delete(1.0, tk.END)
        if self.current_meaning_window:
            self.current_meaning_window.destroy()

        # Extract text from the selected/captured image
        self.extracted_text = self.extract_text_from_image(image_path)

        image = Image.open(image_path)
        self.display_image_on_label(image)

        # Tokenize the extracted text into words
        words_in_extracted_text = word_tokenize(self.extracted_text)

        # Display the extracted words as a paragraph
        for i, word in enumerate(words_in_extracted_text, start=1):
            self.scrolled_text.insert(tk.END, f"{word} ", f"tag_{i}")
            self.scrolled_text.tag_configure(f"tag_{i}", foreground="blue", underline=1)
            self.scrolled_text.tag_bind(f"tag_{i}", "<Button-1>", lambda event, word=word: self.on_word_click(word))

        # Customize the appearance of the scrolled text widget
        self.scrolled_text.configure(bg="white", fg="black", font=("Helvetica", 12))

        # Enable the save buttons after text is extracted
        self.save_word_button.config(state=tk.NORMAL)
        self.save_pdf_button.config(state=tk.NORMAL)

        # Customize the appearance of the import and capture buttons
        self.import_button.configure(bg="#4CAF50", fg="white", font=("Helvetica", 12))
        self.capture_button.configure(bg="#4CAF50", fg="white", font=("Helvetica", 12))
        self.camera_button.configure(bg="#4CAF50", fg="white", font=("Helvetica", 12))

    def on_word_click(self, target_word):
        # Destroy the current meaning window if it exists
        if self.current_meaning_window:
            self.current_meaning_window.destroy()

        # Get the meaning of the clicked word
        meaning = self.get_word_meaning(target_word)

        # Display the meaning in a new window
        meaning_window = tk.Toplevel(self.master)
        meaning_window.title(f"Meaning of '{target_word}'")
        meaning_label = tk.Label(meaning_window, text=meaning, padx=10, pady=10)
        meaning_label.pack()

        # Update the reference to the current meaning window
        self.current_meaning_window = meaning_window

        # Customize the appearance of the meaning window
        meaning_window.configure(bg="#f0f0f0")

        # Customize the appearance of the meaning label
        meaning_label.configure(bg="#f0f0f0", fg="black", font=("Helvetica", 14))

    def show_previous_image(self):
        if self.current_image_index > 0:
            self.current_image_index -= 1
            self.process_image(self.selected_images[self.current_image_index])

    def show_next_image(self):
        if self.current_image_index < len(self.selected_images) - 1:
            self.current_image_index += 1
            self.process_image(self.selected_images[self.current_image_index])

    def save_as_word(self):
        if self.extracted_text:
            save_path = filedialog.asksaveasfilename(defaultextension=".docx", filetypes=[("Word files", "*.docx")])
            if save_path:
                doc = Document()
                doc.add_paragraph(self.extracted_text)
                doc.save(save_path)
                messagebox.showinfo("Success", "Text successfully saved as Word document.")

    def save_as_pdf(self):
        if self.extracted_text:
            save_path = filedialog.asksaveasfilename(defaultextension=".pdf", filetypes=[("PDF files", "*.pdf")])
            if save_path:
                c = canvas.Canvas(save_path, pagesize=letter)
                width, height = letter
                text_lines = self.extracted_text.split('\n')
                y_position = height - 72
                for line in text_lines:
                    c.drawString(72, y_position, line)
                    y_position -= 12  # Move to next line
                c.save()
                messagebox.showinfo("Success", "Text successfully saved as PDF.")


if __name__ == "__main__":
    root = tk.Tk()
    app = WordMeaningApp(root)
    root.mainloop()
